# 🤖 Simple RAG Demo: HR Policy Assistant

**RAG** stands for **Retrieval-Augmented Generation**. In plain English:

1. We take a document (here, an HR policy handbook).
2. We split it into small pieces and convert each piece into a list of numbers (an *embedding*) that captures its meaning.
3. We store those pieces in a searchable database (a *vector store*).
4. When a user asks a question, we **search** the database for the most relevant pieces and pass them to an AI model to **generate** a final answer.

That's it. No magic, just search + a language model.

**What we use in this notebook:**

- 📄 Data: `data/hr_policy.txt` (a sample HR policy document)
- 🔢 Embeddings: **Jina AI**
- 🗄️ Vector store: **FAISS**
- 🧠 LLM: **Groq** (fast + free-tier friendly)
- 🕸️ Framework: **LangChain** (`create_agent`)

> **Before running:** Make sure the notebook's kernel is set to the `ragenv` virtual environment (top-right corner of VS Code / Jupyter), and that a `.env` file containing `GROQ_API_KEY` and `JINA_API_KEY` exists in this folder.

In [1]:
import os 
from dotenv import load_dotenv

In [2]:
# langchain

from langchain_community.document_loaders import TextLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter 

from langchain_community.embeddings import JinaEmbeddings

C:\Users\shara\AppData\Local\Temp\ipykernel_23476\126293049.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = """Rule 1: Always greet the customer with a polite hello.
Rule 2: Never give out internal server passwords.
Rule 3: Escalate to a human if the user gets angry."""

# Setup the smart splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=60,       # Maximum characters allowed per chunk
    chunk_overlap=10     # How many characters to repeat from the previous chunk
)

# Chop the text
chunks = splitter.split_text(text)

# Print the results
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: '{chunk}'")

Chunk 1: 'Rule 1: Always greet the customer with a polite hello.'
Chunk 2: 'Rule 2: Never give out internal server passwords.'
Chunk 3: 'Rule 3: Escalate to a human if the user gets angry.'


In [4]:
groq_key = os.getenv("GROQ_API_KEY")  # LLM 
jina_key = os.getenv("JINA_API_KEY")   # EMBEDDING MODEL

print("ENV VAR LOADED")

ENV VAR LOADED


In [ ]:
# from dotenv import load_dotenv
# import os

# load_dotenv(override=True)

# print("JINA_API_KEY loaded:", bool(os.getenv("JINA_API_KEY")))

In [5]:
DATA_FILE_PATH = os.path.join("data" , "hr_policy.txt")

In [6]:
loader = TextLoader(DATA_FILE_PATH , encoding="utf-8")
documents = loader.load()

In [7]:
documents[0].metadata

{'source': 'data\\hr_policy.txt'}

In [8]:
print(f"Loaded file: {DATA_FILE_PATH}")
print(f"Number of documents loaded: {len(documents)}")
print(f"Total characters in document: {len(documents[0].page_content)}")
print("\n--- Preview of first 300 characters ---")
print(documents[0].page_content[:300])

Loaded file: data\hr_policy.txt
Number of documents loaded: 1
Total characters in document: 2603

--- Preview of first 300 characters ---
COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carr


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [10]:
len(chunks)

9

In [13]:
for chunk in chunks:
    print(chunk)
    print("*"*200)

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)' metadata={'source': 'data\\hr_policy.txt'}
********************************************************************************************************************************************************************************************************
page_content='1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.' metadata={'source': 'data\\hr_policy.txt'}
**************************************************************************************************************************************************************

In [21]:
from langchain_community.embeddings import JinaEmbeddings

embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

In [22]:
embeddings_model.model_name

'jina-embeddings-v2-base-en'

In [23]:
embeddings_model

JinaEmbeddings(session=<requests.sessions.Session object at 0x0000018C57107A10>, model_name='jina-embeddings-v2-base-en', jina_api_key=None)

In [24]:
query_vector = embeddings_model.embed_query("What is LangChain?")

query_vector

[-0.00962079,
 -0.03576203,
 -0.00614623,
 -0.00571117,
 -0.02768769,
 0.01993371,
 -0.00105305,
 -0.02854793,
 0.06249061,
 0.07040081,
 -0.02841345,
 -0.04780927,
 -0.01529932,
 -0.00992187,
 -0.05072023,
 0.09596659,
 -0.01636127,
 -0.01124633,
 0.05107618,
 -0.03327032,
 0.00255549,
 -0.02981356,
 -0.04322135,
 -0.02895135,
 0.07470396,
 0.02174911,
 0.07038499,
 0.05853551,
 0.02928357,
 0.0218638,
 -0.01902799,
 -0.02199037,
 -0.03330196,
 0.00566049,
 0.00278785,
 -0.03365001,
 -0.01911754,
 -0.04297021,
 0.02452163,
 0.07486217,
 -0.04831553,
 -0.01027338,
 0.00664309,
 0.08210792,
 -0.04645663,
 -0.00460572,
 -0.01889748,
 0.02273393,
 -0.05072616,
 0.02377412,
 0.00210819,
 -0.04055957,
 0.03049383,
 -0.05992771,
 0.02606017,
 0.01649475,
 0.05625737,
 -0.04516726,
 -0.02222965,
 -0.0238374,
 -0.01881837,
 0.03407716,
 -0.02861121,
 0.01495622,
 0.01808569,
 0.02791907,
 -0.01657979,
 -0.08058916,
 -0.01703858,
 0.0062629,
 -0.01701633,
 0.0159707,
 0.02659411,
 0.05451712,
 

In [25]:
len(query_vector)

768

In [26]:

from langchain_community.vectorstores import FAISS 

In [27]:
vector_store = FAISS.from_documents(chunks , embeddings_model)

In [28]:
print("CHUNKS ARE STORED" , vector_store.index.ntotal)

CHUNKS ARE STORED 9


In [29]:

test_query = "How many sick leaves employees get"

In [30]:
top_matches = vector_store.similarity_search(test_query , k=3)

In [31]:
top_matches

[Document(id='abac7b61-b0b5-44d1-921c-1240bd224e57', metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'),
 Document(id='1902d7b6-2b9d-4e92-bdb3-ea7714fb5c2c', metadata={'source': 'data\\hr_policy.txt'}, page_content='7. HOLIDAYS\nThe company observes 12 public holidays every year, as per the official holiday calendar\npublished by HR at the start of each year.\nEmployees working on a public holiday are eligible for compensatory leave.'),
 Document(id='d25389d9-290e-4404-a188-b45f7e395bff', metadata={'source': 'data\\hr_policy.txt

In [32]:

print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches,start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()

Query: How many sick leaves employees get

--- Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.

--- Match 3 ---
4. NOTICE PERIOD
Employees who wish to resign must serve a notice period of 30 days.
During the probation period, the notice period is reduced to 15 days.
The company may waive the notice period at its discretion, with full and final se

In [33]:
vector_store.similarity_search

<bound method FAISS.similarity_search of <langchain_community.vectorstores.faiss.FAISS object at 0x0000018C67483770>>

In [34]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})  # returns top 3 relevant chunks


def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

In [41]:
from langchain_groq import ChatGroq 


llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.5
)

In [42]:
answer = llm.invoke("tell me about code of conduct")

In [43]:
print(answer.content)

A **code of conduct** is a set of rules and expectations that a community, organization, or event establishes to guide the behavior of its members.  It’s designed to create a safe, respectful, and inclusive environment where everyone can participate, learn, and contribute without fear of harassment, discrimination, or other negative behaviors.

Below is a general overview of what a typical code of conduct contains, why it matters, and some well‑known examples you might encounter.

---

## 1. Why a Code of Conduct Exists

| Purpose | Why it matters |
|---------|----------------|
| **Safety & Well‑Being** | Protects participants from harassment, hate speech, or any form of intimidation. |
| **Clarity of Expectations** | Provides a shared understanding of acceptable behavior, reducing ambiguity. |
| **Legal & Reputational Protection** | Helps organizations avoid liability and maintain a positive public image. |
| **Inclusivity** | Encourages diverse voices by ensuring everyone feels welco

In [44]:
answer.response_metadata

{'token_usage': {'completion_tokens': 1354,
  'prompt_tokens': 77,
  'total_tokens': 1431,
  'completion_time': 1.548523886,
  'completion_tokens_details': {'reasoning_tokens': 222},
  'prompt_time': 0.003795548,
  'prompt_tokens_details': None,
  'queue_time': 0.278001112,
  'total_time': 1.5523194340000002},
 'model_name': 'openai/gpt-oss-20b',
 'system_fingerprint': 'fp_a8c584dda7',
 'service_tier': 'on_demand',
 'finish_reason': 'stop',
 'logprobs': None,
 'model_provider': 'groq'}

In [45]:

from langchain.agents import create_agent 

In [46]:
from langgraph.checkpoint.memory import InMemorySaver

In [47]:
hr_assistant = create_agent(
    model = llm,
    tools=[search_hr_policy],
    system_prompt= """ 
    
    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    
    always reply in 2 lines , if you are not clear ask a questions to clarift from user
    
    Dont entertain any other questions which are off topic and if you dont understand ,
    tell that mail to acmecorp.com
    """,
    
)

In [48]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages": [{"role": "user",
                            "content": question}]})
    


    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [49]:
response = hr_assistant.invoke({"messages": [{"role": "user",
                            "content": "tell me total number of paid sick leaves"}]})

In [53]:
print(response["messages"][0].content)

tell me total number of paid sick leaves


In [ ]:
print(response["messages"][-1].content)

Paid sick leave entitlement: 10 days per calendar year.  
If you need more details, let me know.


In [56]:

print(ask_hr_assistant("tell me about PROBATION PERIOD"))

QUESTION: tell me about PROBATION PERIOD
------------------------------------------------------------
ANSWER: Probation lasts 3 months from joining; employees cannot take paid leave but may request unpaid leave with manager approval. After probation, a 30‑day notice applies, but during probation it’s reduced to 15 days.

Probation lasts 3 months from joining; employees cannot take paid leave but may request unpaid leave with manager approval. After probation, a 30‑day notice applies, but during probation it’s reduced to 15 days.


In [57]:
print(ask_hr_assistant("what did i ask"))

QUESTION: what did i ask
------------------------------------------------------------
ANSWER: You asked: “what did i ask”.  
Let me know if you need further assistance.

You asked: “what did i ask”.  
Let me know if you need further assistance.


In [58]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

hr_assistant = create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt=""" 
    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know 
    instead of guessing.
    
    Always reply in 2 lines, if you are not clear ask a questions to clarify from user.
    
    Dont entertain any other questions which are off topic and if you dont understand,
    tell that mail to acmecorp.com.
    """,
    checkpointer=InMemorySaver(),
)

# Invoke with a thread_id to maintain memory across turns
config = {"configurable": {"thread_id": "user-session-1"}}

In [59]:
response = hr_assistant.invoke(
    {"messages": [{"role": "user", "content": "What is the that policy?"}]},
    config=config,
)

In [60]:
response["messages"][-1].content

'I’m not sure which policy you’re asking about.  \nCould you clarify which policy you’d like to know more about?'

In [61]:
response = hr_assistant.invoke(
    {"messages": [{"role": "user", "content": "How to make a coffee"}]},
    config=config,
)

In [62]:
response = hr_assistant.invoke(
    {"messages": [{"role": "user", "content": "what was my last question"}]},
    config=config,
)

In [63]:
response["messages"][-1].content

'Your last question was: “How to make a coffee.”  \nLet me know if you’d like a summary of that.'